# 6.2 True duplicate rate

Run this after `group_decision` is filled in `dup_review_50.csv`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv('../data/processed/task6/dup_candidates_6_1.csv')
review = pd.read_csv('../data/processed/task6/dup_review_50.csv')

print("Dataset shape:", df.shape)
print("Review rows:", review.shape)

Dataset shape: (5109, 69)
Review rows: (118, 15)


In [38]:
df_val = pd.read_csv('../data/processed/task6/val_candidates_6_1_val.csv')
print("val rows:", df_val.shape)

val rows: (1252, 69)


In [8]:
(review['group_decision']=='drop').sum()

np.int64(12)

In [9]:
# Validate the manually reviewed decisions
print("Review rows:", len(review))
print("Unique groups:", review["review_no"].nunique())
print("Decision counts:")
print(review["group_decision"].value_counts(dropna=False))

Review rows: 118
Unique groups: 50
Decision counts:
group_decision
keep    106
drop     12
Name: count, dtype: int64


In [11]:
# one decision per group (written in the first row of the group)
decisions = review.dropna(subset=['group_decision']).drop_duplicates('review_no')

print("Groups in sheet:", review['review_no'].nunique())
print("Groups reviewed:", len(decisions))
print(decisions['group_decision'].value_counts())

# stop here if the review is not finished
assert review["group_decision"].notna().all(), \
    "Fill in every row's decision first"

assert set(review["group_decision"].unique()) <= {"drop", "keep"}, \
    "Use only drop / keep"

n_drop = (review["group_decision"] == "drop").sum()
n_keep = (review["group_decision"] == "keep").sum()

print("Ads marked drop:", n_drop)
print("Ads marked keep:", n_keep)


Groups in sheet: 50
Groups reviewed: 50
group_decision
keep    50
Name: count, dtype: int64
Ads marked drop: 12
Ads marked keep: 106


Duplicate Rate in manual review csv

In [ ]:
# drop = 1, keep = 0
score = review['group_decision'].map({'drop': 1,'keep': 0})
drop_rate = score.mean()

margin = 1.96 * np.sqrt(drop_rate * (1 - drop_rate) / len(score))

print(
    f"Share of reviewed ads marked for dropping: "
    f"{drop_rate * 100:.1f}% "
    f"(+/- {margin * 100:.1f})"
)

Share of reviewed ads marked for dropping: 10.2% (+/- 5.5)


# what this means for the whole dataset


In [ ]:
# what this means for the whole dataset
n_candidate_ads = df['dup_candidate_id'].notna().sum()
n_groups = df['dup_candidate_id'].nunique()
extra_ads = n_candidate_ads - n_groups      # ads that would be removed if every group was a duplicate

print("Candidate ads:", n_candidate_ads)
print("Candidate groups:", n_groups)
print("Theoretical extra ads if every group were duplicated:", extra_ads)


Candidate ads: 575 | groups: 238 | extra ads: 337
Estimated duplicate ads in the dataset: 0.7% (between 0.3% and 1.0%)
Apply dedup rule (rate >= 5%): False


# Apply the manual rule to the whole dataset:


In [ ]:
df_check = df.copy()

valid = (df_check["dup_candidate_id"].notna()& df_check["loc_area"].notna())

df_check["drop_by_rule"] = False
df_check.loc[valid, "drop_by_rule"] = (df_check.loc[valid] .duplicated( subset=["dup_candidate_id", "loc_area"], keep="first"))


# 3. Calculate the percentage across the FULL dataset
total_ads = len(df_check)
drop_ads = int(df_check["drop_by_rule"].sum())
keep_ads = total_ads - drop_ads

drop_rate = drop_ads / total_ads * 100

print("\n--- Full Dataset Results ---")
print("Total ads:", total_ads)
print("Ads marked for drop:", drop_ads)
print("Ads kept:", keep_ads)
print(f"Percentage marked for drop: {drop_rate:.2f}%")


Candidate ads: 575
Candidate groups: 238
Theoretical extra ads if every group were duplicated: 337

--- Full Dataset Results ---
Total ads: 5109
Ads marked for drop: 61
Ads kept: 5048
Percentage marked for drop: 1.19%


In [21]:
# does it matter if the storage was observed or imputed?
print(score.groupby(decisions['dup_key_strength']).agg(['mean', 'count']))

                  mean  count
dup_key_strength             
strong             0.0      1
weak               0.0     49


In [24]:

# Summary of the actual deduplication rule applied to the full dataset

total_ads = len(df_check)
drop_ads = int(df_check["drop_by_rule"].sum())
keep_ads = total_ads - drop_ads

dataset_rate = (drop_ads / total_ads)*100
threshold = 0.01
apply_rule = dataset_rate >= threshold

summary = pd.DataFrame([{
    "total_ads": total_ads,
    "candidate_ads": df["dup_candidate_id"].notna().sum(),
    "candidate_groups": df["dup_candidate_id"].nunique(),
    "ads_marked_drop": drop_ads,
    "ads_kept": keep_ads,
    "dataset_drop_rate": round(dataset_rate, 4),
    "threshold": threshold,
    "apply_rule": apply_rule
}])

#summary.to_csv(
#    "../data/processed/task6/dup_rate_6_2.csv",
#    index=False
#)

display(summary)


,total_ads,candidate_ads,candidate_groups,ads_marked_drop,ads_kept,dataset_drop_rate,threshold,apply_rule
0,5109,575,238,61,5048,1.194,0.01,True


In [35]:
df_check.shape

(5109, 70)

# Apply the already-decided drop rule

In [36]:

print("Original dataset:", df_check.shape)

# Keep only ads not marked for dropping
df_cleaned6 = df_check.loc[~df_check["drop_by_rule"]].copy()

print("Ads removed:", len(df_check) - len(df_cleaned6))
print("Cleaned dataset:", df_cleaned6.shape)

# Save without the extra CSV index column
df_cleaned6.to_csv(
    "../data/processed/task6/train_cleaned6.csv",
    index=False
)

print("Saved cleaned dataset successfully.")


Original dataset: (5109, 70)
Ads removed: 61
Cleaned dataset: (5048, 70)
Saved cleaned dataset successfully.


------------------------------------------------

# Apply the manual rule to the validation dataset:


In [39]:
val = df_val.copy()

valid = (val["dup_candidate_id"].notna()& val["loc_area"].notna())

val["drop_by_rule"] = False
val.loc[valid, "drop_by_rule"] = (val.loc[valid] .duplicated( subset=["dup_candidate_id", "loc_area"], keep="first"))


# 3. Calculate the percentage across the FULL dataset
total_ads = len(val)
drop_ads = int(val["drop_by_rule"].sum())
keep_ads = total_ads - drop_ads

drop_rate = drop_ads / total_ads * 100

print("\n--- Full Val Results ---")
print("Total ads:", total_ads)
print("Ads marked for drop:", drop_ads)
print("Ads kept:", keep_ads)
print(f"Percentage marked for drop: {drop_rate:.2f}%")



--- Full Val Results ---
Total ads: 1252
Ads marked for drop: 3
Ads kept: 1249
Percentage marked for drop: 0.24%


In [41]:

print("Original dataset:", val.shape)

# Keep only ads not marked for dropping
df_cleaned6 = val.loc[~val["drop_by_rule"]].copy()

print("Ads removed:", len(val) - len(df_cleaned6))
print("Cleaned dataset:", df_cleaned6.shape)

# Save without the extra CSV index column
df_cleaned6.to_csv(
    "../data/processed/task6/val_cleaned6.csv",
    index=False
)

print("Saved cleaned dataset successfully.")


Original dataset: (1252, 70)
Ads removed: 3
Cleaned dataset: (1249, 70)
Saved cleaned dataset successfully.
